## Udemy course assistent

In [4]:
# imports

import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv("GOOGLE_API_KEY")

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("AQ."):
    print("An API key was found, but it doesn't start AQ.; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=api_key)

API key found and looks good so far!


In [ ]:
# Step 1: Create your prompts

system_prompt = """
    You are an helpful assistant.
    Your task to guide the user to find the best Udemy courses for their career.
    You will be given the user's job title and description.
    You will need to analyze the user's job title and description and recommend the most relevant Udemy courses for them.
    You will need to explain briefly why each course is useful for the user.
    You will need to give the recommended learning order.
    Do not recommend courses just because they are popular. Focus on courses that are relevant to the user's current experience and can help them grow in their career.
    
"""
user_prompt = """
    This is my job title and description:
    Senior Frontend Engineer
    7+ years of experience in React, Typescript and MUI, Rest API.
    Advise me on the best Udemy courses for my career growth.
"""

# Step 2: Make the messages list

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

# Step 3: Call OpenAI
response = gemini.chat.completions.create(
    model="gemini-3.1-flash-lite", 
    messages=messages
)

# Step 4: print the result
Markdown(response.choices[0].message.content)

## Summarize Website 

In [ ]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = """
You are a helpful website summarization assistant.

Summarize only the actual website content provided to you.
Ignore navigation menus, cookie notices, login prompts, advertisements,
and browser/security challenge messages.

If the provided content appears to be a JavaScript, cookie, CAPTCHA,
security, or access-denied page rather than the actual website,
clearly state that the website content could not be retrieved.

Respond in markdown.
Do not wrap the markdown in a code block.
"""

# Define our user prompt

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]


# And now: call the OpenAI API. You will get very familiar with this!

def summarize(url):
    website = fetch_website_contents(url)
    response = gemini.chat.completions.create(
        model="gemini-3.1-flash-lite", 
        messages=messages_for(website)
    )

    return response.choices[0].message.content


# A function to display this nicely in the output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))


display_summary("https://openai.com")